## Transformation of raw data and ingestion into silver layer ##

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("catalog_name","ecommerce","catalog_name")
dbutils.widgets.text("storage_account_name","abiecommerceadlsdev","storage_account_name")
dbutils.widgets.text("container_name","ecomm-raw-data","container_name")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
container_name = dbutils.widgets.get("container_name")

print(catalog_name, storage_account_name, container_name)

In [0]:
silver_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoints/silver/fact_order_returns"

In [0]:
df_bronze_returns = spark.readStream.format("delta").table(f"{catalog_name}.bronze.brz_order_returns")


In [0]:
df_silver_returns = df_bronze_returns.dropDuplicates(["order_id","order_dt", "return_ts"])

In [0]:
df_silver_returns = df_silver_returns.withColumn("order_dt", df_silver_returns["order_dt"].cast("date"))

In [0]:
df_silver_returns = df_silver_returns.withColumn("return_ts", df_silver_returns["return_ts"].cast("timestamp"))
df_silver_returns = df_silver_returns.withColumn("reason", F.trim(F.upper(df_silver_returns["reason"])))

In [0]:
df_silver_returns = df_silver_returns.withColumn("processed_time", F.current_timestamp())


In [0]:
def upsert_to_silver(microBatchDF, batchId):
    table_name = f"{catalog_name}.silver.slv_order_returns"
    if not spark.catalog.tableExists(table_name):
        print("creating new table")
        microBatchDF.write.format("delta").mode("overwrite").saveAsTable(table_name)
        spark.sql(
            f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
        )
    else:
        deltaTable = DeltaTable.forName(spark, table_name)
        deltaTable.alias("silver_table").merge(
            microBatchDF.alias("batch_table"),
            "silver_table.order_id = batch_table.order_id AND silver_table.order_dt = batch_table.order_dt AND silver_table.return_ts = batch_table.return_ts"
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()    


In [0]:
df_silver_returns.writeStream.trigger(availableNow=True).foreachBatch(
    upsert_to_silver
).format("delta").option("checkpointLocation", silver_checkpoint_path).option(
    "mergeSchema", "true"
).outputMode(
    "update"
).trigger(
    once=True
).start().awaitTermination()